In [ ]:
# =========================================
# 🧩 Deepfake Detection with EfficientNet, SE, and Attention Modules
# Dataset: WildDeepfake (KaggleHub)
# =========================================

import os
import zipfile
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import kagglehub
import tensorflow as tf
from sklearn.metrics import classification_report, confusion_matrix, roc_curve, auc
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import EfficientNetB0
from tensorflow.keras.layers import GlobalAveragePooling2D, Dense, Dropout, Multiply, Reshape, Conv2D, Add, Lambda, Concatenate
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint

# =========================================
# 1️⃣ Download Dataset
# =========================================
print("⬇️ Downloading WildDeepfake dataset...")
download_path = kagglehub.dataset_download("maysuni/wild-deepfake")
print("✅ Dataset downloaded at:", download_path)

# Unzip if necessary
dataset_path = download_path
for file in os.listdir(download_path):
    if file.endswith(".zip"):
        zip_path = os.path.join(download_path, file)
        extract_dir = os.path.join(download_path, "WildDeepfake")
        if not os.path.exists(extract_dir):
            with zipfile.ZipFile(zip_path, 'r') as zip_ref:
                zip_ref.extractall(extract_dir)
        dataset_path = extract_dir
        break

# Verify the path and find the correct directory containing 'fake' and 'real' folders
if not os.path.exists(os.path.join(dataset_path, "fake")) or not os.path.exists(os.path.join(dataset_path, "real")):
    print("Searching for 'fake' and 'real' directories within the downloaded path...")
    found_path = None
    for root, dirs, files in os.walk(download_path):
        if 'fake' in dirs and 'real' in dirs:
            found_path = root
            break
    if found_path:
        dataset_path = found_path
        print(f"Found 'fake' and 'real' directories at: {dataset_path}")
    else:
        raise FileNotFoundError("Could not find 'fake' and 'real' directories within the downloaded dataset.")

print("Using dataset path:", dataset_path)

# =========================================
# 2️⃣ Data Preprocessing and Augmentation
# =========================================
img_size = (224, 224)
batch_size = 32

train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=10,
    width_shift_range=0.1,
    height_shift_range=0.1,
    shear_range=0.1,
    zoom_range=0.1,
    brightness_range=[0.8, 1.2],
    horizontal_flip=True,
    fill_mode='nearest',
    validation_split=0.2
)

val_datagen = ImageDataGenerator(rescale=1./255, validation_split=0.2)

train_gen = train_datagen.flow_from_directory(
    dataset_path,
    target_size=img_size,
    batch_size=batch_size,
    class_mode='binary',
    subset='training'
)

val_gen = val_datagen.flow_from_directory(
    dataset_path,
    target_size=img_size,
    batch_size=batch_size,
    class_mode='binary',
    subset='validation'
)

⬇️ Downloading WildDeepfake dataset...
✅ Dataset downloaded at: /root/.cache/kagglehub/datasets/maysuni/wild-deepfake/versions/1
Searching for 'fake' and 'real' directories within the downloaded path...
Found 'fake' and 'real' directories at: /root/.cache/kagglehub/datasets/maysuni/wild-deepfake/versions/1/valid
Using dataset path: /root/.cache/kagglehub/datasets/maysuni/wild-deepfake/versions/1/valid
Found 13879 images belonging to 2 classes.
Found 3469 images belonging to 2 classes.


In [ ]:

# =========================================
# 3️⃣ Define Modules
# =========================================
def squeeze_excite_block(inputs, ratio=16):
    """Squeeze-and-Excitation block"""
    filters = inputs.shape[-1]
    se = GlobalAveragePooling2D()(inputs)
    se = Dense(filters // ratio, activation='relu')(se)
    se = Dense(filters, activation='sigmoid')(se)
    se = Reshape([1, 1, filters])(se)
    x = Multiply()([inputs, se])
    return x


def attention_block(inputs):
    """CBAM-like Attention block"""
    # Channel attention
    avg_pool = GlobalAveragePooling2D()(inputs)
    max_pool = tf.reduce_max(inputs, axis=[1, 2])
    concat = Add()([
        Dense(inputs.shape[-1] // 8, activation='relu')(avg_pool),
        Dense(inputs.shape[-1] // 8, activation='relu')(max_pool)
    ])
    channel_attention = Dense(inputs.shape[-1], activation='sigmoid')(concat)
    channel_attention = Reshape([1, 1, inputs.shape[-1]])(channel_attention)
    x = Multiply()([inputs, channel_attention])

    # Spatial attention
    avg_pool = Lambda(lambda x: tf.reduce_mean(x, axis=-1, keepdims=True))(x)
    max_pool = Lambda(lambda x: tf.reduce_max(x, axis=-1, keepdims=True))(x)
    concat = Concatenate(axis=-1)([avg_pool, max_pool])
    spatial_attention = Conv2D(1, (7, 7), activation='sigmoid', padding='same')(concat)
    out = Multiply()([x, spatial_attention])
    return out


In [ ]:

# =========================================
# 4️⃣ Model Builder
# =========================================
def build_model(name, use_se=False, use_attention=False):
    base = EfficientNetB0(weights='imagenet', include_top=False, input_shape=(224, 224, 3))
    x = base.output

    if use_se:
        x = squeeze_excite_block(x)

    if use_attention:
        x = attention_block(x)

    x = GlobalAveragePooling2D()(x)
    x = Dropout(0.3)(x)
    output = Dense(1, activation='sigmoid')(x)

    model = Model(inputs=base.input, outputs=output, name=name)
    model.compile(optimizer=Adam(1e-4), loss='binary_crossentropy', metrics=['accuracy'])
    return model


In [ ]:

# =========================================
# 5️⃣ Model Variants
# =========================================
models_config = [
    ("M1_EfficientNet", False, False),
    ("M2_EfficientNet_SE", True, False),
    ("M3_EfficientNet_Att", False, True),
    ("M4_SE_Att", True, True),
    ("M5_EfficientNet_SE_Att", True, True)
]

history_dict = {}


In [ ]:

# =========================================
# 6️⃣ Training Loop
# =========================================
for name, se, att in models_config:
    print(f"\n🚀 Training {name}")
    model = build_model(name, use_se=se, use_attention=att)
    checkpoint = ModelCheckpoint(f"{name}.h5", save_best_only=True, monitor='val_accuracy', mode='max')
    early_stop = EarlyStopping(patience=5, restore_best_weights=True)

    history = model.fit(
        train_gen,
        validation_data=val_gen,
        epochs=20,
        callbacks=[checkpoint, early_stop],
        verbose=1
    )
    history_dict[name] = history.history

# =========================================
# 7️⃣ Evaluation Function
# =========================================
def evaluate_model(model, generator):
    y_true = generator.classes
    y_pred = model.predict(generator).ravel()
    y_pred_classes = (y_pred > 0.5).astype(int)

    print(classification_report(y_true, y_pred_classes))

    cm = confusion_matrix(y_true, y_pred_classes)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
    plt.title("Confusion Matrix")
    plt.show()

    fpr, tpr, _ = roc_curve(y_true, y_pred)
    roc_auc = auc(fpr, tpr)
    plt.plot(fpr, tpr, label=f"AUC = {roc_auc:.2f}")
    plt.legend()
    plt.title("ROC Curve")
    plt.show()

# =========================================
# 8️⃣ Evaluate Each Trained Model
# =========================================
for name, _, _ in models_config:
    print(f"\n🔍 Evaluating {name}")
    model = tf.keras.models.load_model(f"{name}.h5", compile=False)
    model.compile(optimizer=Adam(1e-4), loss='binary_crossentropy', metrics=['accuracy'])
    evaluate_model(model, val_gen)

print("✅ All models trained and evaluated successfully!")


🚀 Training M1_EfficientNet
16705208/16705208 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


/usr/local/lib/python3.12/dist-packages/keras/src/trainers/data_adapters/py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


Epoch 1/20
434/434 ━━━━━━━━━━━━━━━━━━━━ 0s 585ms/step - accuracy: 0.8016 - loss: 0.4069

434/434 ━━━━━━━━━━━━━━━━━━━━ 362s 645ms/step - accuracy: 0.8018 - loss: 0.4066 - val_accuracy: 0.4514 - val_loss: 0.8301
Epoch 2/20
434/434 ━━━━━━━━━━━━━━━━━━━━ 0s 478ms/step - accuracy: 0.9691 - loss: 0.0898

434/434 ━━━━━━━━━━━━━━━━━━━━ 217s 500ms/step - accuracy: 0.9691 - loss: 0.0898 - val_accuracy: 0.4875 - val_loss: 231.4486
Epoch 3/20
434/434 ━━━━━━━━━━━━━━━━━━━━ 0s 472ms/step - accuracy: 0.9800 - loss: 0.0549

...

434/434 ━━━━━━━━━━━━━━━━━━━━ 216s 498ms/step - accuracy: 0.9872 - loss: 0.0383 - val_accuracy: 0.6351 - val_loss: 0.6847
Epoch 5/20
434/434 ━━━━━━━━━━━━━━━━━━━━ 212s 488ms/step - accuracy: 0.9916 - loss: 0.0229 - val_accuracy: 0.4961 - val_loss: 548.3646
Epoch 6/20
434/434 ━━━━━━━━━━━━━━━━━━━━ 0s 463ms/step - accuracy: 0.9928 - loss: 0.0206

434/434 ━━━━━━━━━━━━━━━━━━━━ 210s 484ms/step - accuracy: 0.9928 - loss: 0.0206 - val_accuracy: 0.6463 - val_loss: 0.6878
Epoch 7/20
434/434 ━━━━━━━━━━━━━━━━━━━━ 215s 495ms/step - accuracy: 0.9924 - loss: 0.0203 - val_accuracy: 0.5267 - val_loss: 0.7853
Epoch 8/20
434/434 ━━━━━━━━━━━━━━━━━━━━ 215s 496ms/step - accuracy: 0.9925 - loss: 0.0195 - val_accuracy: 0.4581 - val_loss: 1.0274
Epoch 9/20
434/434 ━━━━━━━━━━━━━━━━━━━━ 219s 504ms/step - accuracy: 0.9921 - loss: 0.0198 - val_accuracy: 0.5523 - val_loss: 0.8181

🚀 Training M2_EfficientNet_SE
Epoch 1/20
434/434 ━━━━━━━━━━━━━━━━━━━━ 0s 564ms/step - accuracy: 0.7774 - loss: 0.4459

434/434 ━━━━━━━━━━━━━━━━━━━━ 335s 612ms/step - accuracy: 0.7776 - loss: 0.4455 - val_accuracy: 0.6466 - val_loss: 0.9353
Epoch 2/20
434/434 ━━━━━━━━━━━━━━━━━━━━ 214s 493ms/step - accuracy: 0.9627 - loss: 0.0926 - val_accuracy: 0.4719 - val_loss: 1.1262
Epoch 3/20
434/434 ━━━━━━━━━━━━━━━━━━━━ 210s 484ms/step - accuracy: 0.9781 - loss: 0.0557 - val_accuracy: 0.4875 - val_loss: 459.7592
Epoch 4/20
434/434 ━━━━━━━━━━━━━━━━━━━━ 210s 484ms/step - accuracy: 0.9848 - loss: 0.0420 - val_accuracy: 0.5457 - val_loss: 0.9101
Epoch 5/20
434/434 ━━━━━━━━━━━━━━━━━━━━ 215s 495ms/step - accuracy: 0.9885 - loss: 0.0314 - val_accuracy: 0.5365 - val_loss: 0.8416
Epoch 6/20
434/434 ━━━━━━━━━━━━━━━━━━━━ 210s 483ms/step - accuracy: 0.9901 - loss: 0.0252 - val_accuracy: 0.5391 - val_loss: 0.9273
Epoch 7/20
434/434 ━━━━━━━━━━━━━━━━━━━━ 213s 490ms/step - accuracy: 0.9930 - loss: 0.0184 - val_accuracy: 0.5915 - val_loss: 0.8211
Epoch 8/20
434/434 ━━━━━━━━━━━━━━━━━━━━ 214s 493ms/step - accuracy: 0.9937 - loss: 0.

434/434 ━━━━━━━━━━━━━━━━━━━━ 214s 494ms/step - accuracy: 0.9951 - loss: 0.0126 - val_accuracy: 0.6498 - val_loss: 0.6278
Epoch 12/20
434/434 ━━━━━━━━━━━━━━━━━━━━ 215s 494ms/step - accuracy: 0.9957 - loss: 0.0107 - val_accuracy: 0.4644 - val_loss: 521.2910
Epoch 13/20
434/434 ━━━━━━━━━━━━━━━━━━━━ 212s 489ms/step - accuracy: 0.9953 - loss: 0.0109 - val_accuracy: 0.5402 - val_loss: 0.8541
Epoch 14/20
434/434 ━━━━━━━━━━━━━━━━━━━━ 216s 498ms/step - accuracy: 0.9961 - loss: 0.0107 - val_accuracy: 0.4932 - val_loss: 0.7214
Epoch 15/20
434/434 ━━━━━━━━━━━━━━━━━━━━ 212s 489ms/step - accuracy: 0.9959 - loss: 0.0084 - val_accuracy: 0.5448 - val_loss: 221.1522
Epoch 16/20
434/434 ━━━━━━━━━━━━━━━━━━━━ 210s 483ms/step - accuracy: 0.9968 - loss: 0.0081 - val_accuracy: 0.4707 - val_loss: 1.5845

🚀 Training M3_EfficientNet_Att


ValueError: A KerasTensor cannot be used as input to a TensorFlow function. A KerasTensor is a symbolic placeholder for a shape and dtype, used when constructing Keras Functional models or Keras Functions. You can only use it as input to a Keras layer or a Keras operation (from the namespaces `keras.layers` and `keras.ops`). You are likely doing something like:

```
x = Input(...)
...
tf_fn(x)  # Invalid.
```

What you should do instead is wrap `tf_fn` in a layer:

```
class MyLayer(Layer):
    def call(self, x):
        return tf_fn(x)

x = MyLayer()(x)
```
